In [1]:
import pandas as pd

splits = {'train': 'train.jsonl.gz', 'validation': 'validation.jsonl.gz', 'test': 'test.jsonl.gz'}
df = pd.read_json("hf://datasets/mteb/stsbenchmark-sts/" + splits["train"], lines=True)

/Users/macbookpro/case_study/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df

,split,genre,dataset,year,sid,score,sentence1,sentence2
0,train,main-captions,MSRvid,2012test,1,5.00,A plane is taking off.,An air plane is taking off.
1,train,main-captions,MSRvid,2012test,4,3.80,A man is playing a large flute.,A man is playing a flute.
2,train,main-captions,MSRvid,2012test,5,3.80,A man is spreading shreded cheese on a pizza.,A man is spreading shredded cheese on an uncoo...
3,train,main-captions,MSRvid,2012test,6,2.60,Three men are playing chess.,Two men are playing chess.
4,train,main-captions,MSRvid,2012test,9,4.25,A man is playing the cello.,A man seated is playing the cello.
...,...,...,...,...,...,...,...,...
5744,train,main-news,headlines,2016,1456,0.00,Severe Gales As Storm Clodagh Hits Britain,Merkel pledges NATO solidarity with Latvia
5745,train,main-news,headlines,2016,1465,0.00,Dozens of Egyptians hostages taken by Libyan t...,Egyptian boat crash death toll rises as more b...
5746,train,main-news,headlines,2016,1466,0.00,President heading to Bahrain,President Xi: China to continue help to fight ...
5747,train,main-news,headlines,2016,1470,0.00,"China, India vow to further bilateral ties",China Scrambles to Reassure Jittery Stock Traders


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5749 entries, 0 to 5748
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   split      5749 non-null   object 
 1   genre      5749 non-null   object 
 2   dataset    5749 non-null   object 
 3   year       5749 non-null   object 
 4   sid        5749 non-null   int64  
 5   score      5749 non-null   float64
 6   sentence1  5749 non-null   object 
 7   sentence2  5749 non-null   object 
dtypes: float64(1), int64(1), object(6)
memory usage: 359.4+ KB


In [4]:
# check range of score values
print(df['score'].min())
print(df['score'].max())

0.0
5.0


In [5]:
from datasets import Dataset

In [6]:
dataset = Dataset.from_pandas(df[['score', 'sentence1', 'sentence2']])
dataset

Dataset({
    features: ['score', 'sentence1', 'sentence2'],
    num_rows: 5749
})

# Static - word2vec

In [7]:
import gensim.downloader as api

word2vec_model = api.load("word2vec-google-news-300")

In [8]:
from nltk.tokenize import word_tokenize
import numpy as np

dim = word2vec_model.vector_size

def vectorize_and_get_similarity_static(example):
    sent1, sent2 = example['sentence1'].lower(), example['sentence2'].lower()
    # tokenization of sentences
    tokenized1 = word_tokenize(sent1)
    tokenized2 = word_tokenize(sent2)
    # compute vectors
    vec1 = np.mean([word2vec_model[token].astype(np.float32) if token in word2vec_model else np.zeros(dim, dtype=np.float32) \
                         for token in tokenized1], axis=0)
    vec2 = np.mean([word2vec_model[token].astype(np.float32) if token in word2vec_model else np.zeros(dim, dtype=np.float32) \
                         for token in tokenized2], axis=0)
    # scaling to the score range [0, 5]
    similarity = 5 * np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
    return {'vec1': vec1, 'vec2': vec2, 'score': example['score'], 'similarity': similarity}

static_vectors = dataset.map(vectorize_and_get_similarity_static, remove_columns=dataset.features.keys())

Map: 100%|█████████████████████████| 5749/5749 [00:01<00:00, 5148.85 examples/s]


In [9]:
static_vectors

Dataset({
    features: ['score', 'vec1', 'vec2', 'similarity'],
    num_rows: 5749
})

In [10]:
from scipy.stats import pearsonr, spearmanr

pearson_stat, _ = pearsonr(static_vectors['score'], static_vectors['similarity'])
spearman_stat, _ = spearmanr(static_vectors['score'], static_vectors['similarity'])

print("Pearson:", pearson_stat)
print("Spearman:", spearman_stat)

Pearson: 0.684072261537721
Spearman: 0.6387639851665995


**Word2Vec** achieved a reasonably strong correlation with human judgments (**Pearson = 0.684**, **Spearman = 0.639**). This indicates that static embeddings capture general semantic relatedness fairly well, although they often struggle with contextual nuances and polysemy.

# Contextual embeddings - Bert

In [11]:
from transformers import AutoTokenizer, AutoModel

tokenizer_bert = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
model_bert = AutoModel.from_pretrained("google-bert/bert-base-uncased")

In [12]:
import torch
import numpy as np
from functools import partial

def vectorize_and_get_similarity_contextual(example, model, tokenizer):
    sent1, sent2 = example['sentence1'], example['sentence2']
    inputs1 = tokenizer(sent1, return_tensors='pt')
    inputs2 = tokenizer(sent2, return_tensors='pt')
    
    with torch.no_grad():
        vec1 = model(**inputs1).last_hidden_state.mean(dim=1)[0]
        vec2 = model(**inputs2).last_hidden_state.mean(dim=1)[0]

    similarity = 5 * np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
    return {'vec1': vec1, 'vec2': vec2, 'score': example['score'], 'similarity': similarity}

vectorize_func_bert = partial(vectorize_and_get_similarity_contextual, model=model_bert, tokenizer=tokenizer_bert)
contextual_vectors_bert = dataset.map(vectorize_func_bert, remove_columns=dataset.features.keys())

Map: 100%|███████████████████████████| 5749/5749 [04:20<00:00, 22.08 examples/s]


In [13]:
contextual_vectors_bert

Dataset({
    features: ['score', 'vec1', 'vec2', 'similarity'],
    num_rows: 5749
})

In [14]:
pearson_cntx_bert, _ = pearsonr(contextual_vectors_bert['score'], contextual_vectors_bert['similarity'])
spearman_cntx_bert, _ = spearmanr(contextual_vectors_bert['score'], contextual_vectors_bert['similarity'])

print("Pearson:", pearson_cntx_bert)
print("Spearman:", spearman_cntx_bert)

Pearson: 0.5113658172817259
Spearman: 0.48385206304632883


**BERT** showed noticeably weaker performance (**Pearson = 0.511**, **Spearman = 0.484**). Although **BERT** produces contextual token representations, simple mean pooling is not optimal for sentence similarity tasks, which explains its relatively low alignment with human annotations.

# Contextual embeddings - Sentence Transformers

In [15]:
from transformers import AutoTokenizer, AutoModel

tokenizer_st = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model_st = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

In [16]:
vectorize_func_st = partial(vectorize_and_get_similarity_contextual, model=model_st, tokenizer=tokenizer_st)
contextual_vectors_st = dataset.map(vectorize_func_st, remove_columns=dataset.features.keys())

Map: 100%|██████████████████████████| 5749/5749 [00:52<00:00, 108.62 examples/s]


In [17]:
pearson_cntx_st, _ = pearsonr(contextual_vectors_st['score'], contextual_vectors_st['similarity'])
spearman_cntx_st, _ = spearmanr(contextual_vectors_st['score'], contextual_vectors_st['similarity'])

print("Pearson:", pearson_cntx_st)
print("Spearman:", spearman_cntx_st)

Pearson: 0.8348826558429163
Spearman: 0.8100235305483507


**Sentence-BERT** delivered the best results by a substantial margin (**Pearson = 0.835**, **Spearman = 0.810**). Its architecture is specifically optimized for sentence-level semantic similarity, making it highly effective at capturing both semantic content and relative ranking.

# Performance differences analysis

## 1. Lexical Overlap (Surface-Level Similarity)

Lexical overlap refers to sentence pairs that share many identical or highly similar words while expressing substantially different meanings. These examples are particularly challenging for static embedding models, which often rely heavily on shared vocabulary and may fail to capture crucial semantic distinctions such as word order, semantic roles, or entity substitutions.

Such cases are especially important in Semantic Textual Similarity because high lexical similarity does not necessarily imply high semantic similarity.

In [18]:
# these examples were taken from df
lexical_overlaps = [
    {'id': 73, 'sentence1': 'A man is playing the piano.', 'sentence2': 'A man is playing the trumpet.', 'score': 1.600},
    {'id': 2107, 'sentence1': 'There is no freedom of religion.', 'sentence2': 'There is no freedom of speech.', 'score': 1.200},
    {'id': 2424, 'sentence1': 'Bush gets zero credit for the budget under him.', 'sentence2': 'And, Obama gets zero credit for the budget under him.', 'score': 1.800},
    {'id': 3638, 'sentence1': 'zahedan is a city in southeastern', 'sentence2': 'bam is a city in southeastern', 'score': 1.800}
]

lexical_overlaps_dataset = Dataset.from_list(lexical_overlaps)
pd.DataFrame(lexical_overlaps)

,id,sentence1,sentence2,score
0,73,A man is playing the piano.,A man is playing the trumpet.,1.6
1,2107,There is no freedom of religion.,There is no freedom of speech.,1.2
2,2424,Bush gets zero credit for the budget under him.,"And, Obama gets zero credit for the budget und...",1.8
3,3638,zahedan is a city in southeastern,bam is a city in southeastern,1.8


#### • Evaluation with `word2vec-google-news-300` (static)

In [19]:
lexical_overlaps_dataset.map(vectorize_and_get_similarity_static).to_pandas().drop(['vec1', 'vec2'], axis=1)

Map: 100%|████████████████████████████████| 4/4 [00:00<00:00, 452.48 examples/s]


,id,sentence1,sentence2,score,similarity
0,73,A man is playing the piano.,A man is playing the trumpet.,1.6,4.488122
1,2107,There is no freedom of religion.,There is no freedom of speech.,1.2,4.290080
2,2424,Bush gets zero credit for the budget under him.,"And, Obama gets zero credit for the budget und...",1.8,4.513772
3,3638,zahedan is a city in southeastern,bam is a city in southeastern,1.8,4.241587


#### • Evaluation with `google-bert/bert-base-uncased` (contextual)

In [20]:
lexical_overlaps_dataset.map(vectorize_func_bert).to_pandas().drop(['vec1', 'vec2'], axis=1)

Map: 100%|█████████████████████████████████| 4/4 [00:00<00:00, 25.85 examples/s]


,id,sentence1,sentence2,score,similarity
0,73,A man is playing the piano.,A man is playing the trumpet.,1.6,4.773473
1,2107,There is no freedom of religion.,There is no freedom of speech.,1.2,4.675400
2,2424,Bush gets zero credit for the budget under him.,"And, Obama gets zero credit for the budget und...",1.8,4.409699
3,3638,zahedan is a city in southeastern,bam is a city in southeastern,1.8,4.135039


#### • Evaluation with `sentence-transformers/all-MiniLM-L6-v2` (contextual)

In [21]:
lexical_overlaps_dataset.map(vectorize_func_st).to_pandas().drop(['vec1', 'vec2'], axis=1)

Map: 100%|████████████████████████████████| 4/4 [00:00<00:00, 102.88 examples/s]


,id,sentence1,sentence2,score,similarity
0,73,A man is playing the piano.,A man is playing the trumpet.,1.6,2.652764
1,2107,There is no freedom of religion.,There is no freedom of speech.,1.2,3.798497
2,2424,Bush gets zero credit for the budget under him.,"And, Obama gets zero credit for the budget und...",1.8,4.500367
3,3638,zahedan is a city in southeastern,bam is a city in southeastern,1.8,1.465667


Among the evaluated models, **Word2Vec** and **BERT** consistently assigned very high similarity scores to all sentence pairs, despite their relatively low human similarity annotations. This indicates that both models are highly sensitive to shared vocabulary and sentence structure. In particular, they struggle to recognize that replacing a single important word (such as piano → trumpet or religion → speech) can substantially alter meaning.
**Sentence-BERT (all-MiniLM-L6-v2)** demonstrated noticeably better robustness. Its scores were generally lower and closer to human judgments, especially for examples where only one semantically crucial word changed. This suggests that **Sentence-BERT** captures sentence-level semantics more effectively and is less biased by superficial lexical overlap.
Overall, lexical overlap remains a difficult phenomenon for embedding models. Static embeddings like **Word2Vec** are the most vulnerable, contextual encoders like **BERT** improve only slightly, while sentence-level transformers provide the most reliable semantic discrimination in such cases.

## 2. Paraphrases

Paraphrases are sentence pairs that convey the same meaning using different words, grammatical structures, or syntactic constructions. They represent one of the core challenges in semantic similarity tasks, requiring models to move beyond simple lexical matching.

High-quality sentence embeddings should assign very high similarity scores to paraphrastic pairs.

In [22]:
# these examples were taken from df
paraphrases = [
    {'id': 247, 'sentence1': 'The boy is playing the piano.', 'sentence2': 'A boy plays a piano.', 'score': 5},
    {'id': 377, 'sentence1': 'A group of people sing.', 'sentence2': 'Some people are singing.', 'score': 5},
    {'id': 716, 'sentence1': 'A man is firing a shot gun.', 'sentence2': 'A man shoots a shotgun.', 'score': 5},
    {'id': 973, 'sentence1': 'The man is using a sledghammer to break the concrete block that is on the other man.', 
         'sentence2': 'A man breaks a slab of concrete that is lying on a prone man with a sledge hammer.', 'score': 5},
    {'id': 5605, 'sentence1': 'Spain approves new restrictive abortion law', 
         'sentence2': 'Spanish government approves tight restrictions on abortion', 'score': 5},
    {'id': 5431, 'sentence1': 'Protesters gather in Brazil despite concessions', 
         'sentence2': 'Protests sweep Brazil despite concession', 'score': 5}
]

paraphrases_dataset = Dataset.from_list(paraphrases)
pd.DataFrame(paraphrases)

,id,sentence1,sentence2,score
0,247,The boy is playing the piano.,A boy plays a piano.,5
1,377,A group of people sing.,Some people are singing.,5
2,716,A man is firing a shot gun.,A man shoots a shotgun.,5
3,973,The man is using a sledghammer to break the co...,A man breaks a slab of concrete that is lying ...,5
4,5605,Spain approves new restrictive abortion law,Spanish government approves tight restrictions...,5
5,5431,Protesters gather in Brazil despite concessions,Protests sweep Brazil despite concession,5


#### • Evaluation with `word2vec-google-news-300` (static)

In [38]:
paraphrases_dataset.map(vectorize_and_get_similarity_static).to_pandas().drop(['vec1', 'vec2'], axis=1)

Map: 100%|████████████████████████████████| 6/6 [00:00<00:00, 682.91 examples/s]


,id,sentence1,sentence2,score,similarity
0,247,The boy is playing the piano.,A boy plays a piano.,5,4.065207
1,377,A group of people sing.,Some people are singing.,5,3.633395
2,716,A man is firing a shot gun.,A man shoots a shotgun.,5,3.733452
3,973,The man is using a sledghammer to break the co...,A man breaks a slab of concrete that is lying ...,5,3.842156
4,5605,Spain approves new restrictive abortion law,Spanish government approves tight restrictions...,5,4.033671
5,5431,Protesters gather in Brazil despite concessions,Protests sweep Brazil despite concession,5,3.824014


#### • Evaluation with `google-bert/bert-base-uncased` (contextual)

In [24]:
paraphrases_dataset.map(vectorize_func_bert).to_pandas().drop(['vec1', 'vec2'], axis=1)

Map: 100%|█████████████████████████████████| 6/6 [00:00<00:00, 25.03 examples/s]


,id,sentence1,sentence2,score,similarity
0,247,The boy is playing the piano.,A boy plays a piano.,5,4.419295
1,377,A group of people sing.,Some people are singing.,5,4.406513
2,716,A man is firing a shot gun.,A man shoots a shotgun.,5,4.650848
3,973,The man is using a sledghammer to break the co...,A man breaks a slab of concrete that is lying ...,5,4.357868
4,5605,Spain approves new restrictive abortion law,Spanish government approves tight restrictions...,5,4.626211
5,5431,Protesters gather in Brazil despite concessions,Protests sweep Brazil despite concession,5,4.486828


#### • Evaluation with `sentence-transformers/all-MiniLM-L6-v2` (contextual)

In [25]:
paraphrases_dataset.map(vectorize_func_st).to_pandas().drop(['vec1', 'vec2'], axis=1)

Map: 100%|████████████████████████████████| 6/6 [00:00<00:00, 112.37 examples/s]


,id,sentence1,sentence2,score,similarity
0,247,The boy is playing the piano.,A boy plays a piano.,5,4.721477
1,377,A group of people sing.,Some people are singing.,5,4.533153
2,716,A man is firing a shot gun.,A man shoots a shotgun.,5,4.052787
3,973,The man is using a sledghammer to break the co...,A man breaks a slab of concrete that is lying ...,5,3.550198
4,5605,Spain approves new restrictive abortion law,Spanish government approves tight restrictions...,5,4.382178
5,5431,Protesters gather in Brazil despite concessions,Protests sweep Brazil despite concession,5,3.741496


Among the evaluated models, **BERT (bert-base-uncased)** demonstrates the strongest and most consistent performance overall. Its similarity scores remain uniformly high across both simple and more complex paraphrase pairs, closely aligning with the maximum human score of 5.0. This suggests that BERT effectively captures contextual meaning while maintaining robustness across diverse paraphrasing patterns.

**Sentence-BERT (all-MiniLM-L6-v2)** also performs very well, particularly on shorter and more direct paraphrases. However, its scores show greater variability on longer or more lexically diverse examples, where it occasionally underestimates semantic equivalence.

**Word2Vec**, while capable of identifying obvious paraphrases, consistently produces lower similarity values. Its reliance on static word embeddings limits its ability to fully account for syntactic transformations and contextual nuances.

Overall, paraphrase detection appears to be best handled by contextual models. In this evaluation, **BERT slightly outperforms Sentence-BERT**, especially on more challenging examples, while Word2Vec remains the least accurate of the three.

## 3. Polysemy

Polysemy occurs when a single word has multiple meanings depending on context. For example, the word bank may refer to a financial institution or the side of a river.

Polysemous words present a major challenge for static embeddings because a single vector must represent all possible senses.

> **Remark:** `Since explicit polysemous sentence pairs are rare in dataset, manually curated examples involving ambiguous words such as "bank", "bat", "light" and others were used for qualitative analysis.`

In [26]:
# these examples were NOT taken from df
polysemy = [
    {'sentence1': 'I deposited money at the bank.', 'sentence2': 'We sat on the bank of the river.', 'score': None},
    {'sentence1': 'A bat flew out of the cave.', 'sentence2': 'He swung the bat and hit the ball.', 'score': None},
    {'sentence1': 'Please turn on the light.', 'sentence2': 'This suitcase is very light.', 'score': None},
    {'sentence1': 'She struck a match.', 'sentence2': 'The football match was exciting.', 'score': None},
    {'sentence1': 'The store will charge your credit card.', 'sentence2': 'The soldier began to charge forward.', 'score': None},
    {'sentence1': 'Flowers bloom in spring.', 'sentence2': 'The mattress has a broken spring.', 'score': None}
]

polysemy_dataset = Dataset.from_list(polysemy)
pd.DataFrame(polysemy)

,sentence1,sentence2,score
0,I deposited money at the bank.,We sat on the bank of the river.,None
1,A bat flew out of the cave.,He swung the bat and hit the ball.,None
2,Please turn on the light.,This suitcase is very light.,None
3,She struck a match.,The football match was exciting.,None
4,The store will charge your credit card.,The soldier began to charge forward.,None
5,Flowers bloom in spring.,The mattress has a broken spring.,None


#### • Evaluation with `word2vec-google-news-300` (static)

In [27]:
polysemy_dataset.map(vectorize_and_get_similarity_static).to_pandas().drop(['vec1', 'vec2'], axis=1)

Map: 100%|████████████████████████████████| 6/6 [00:00<00:00, 873.24 examples/s]


,sentence1,sentence2,score,similarity
0,I deposited money at the bank.,We sat on the bank of the river.,None,2.927340
1,A bat flew out of the cave.,He swung the bat and hit the ball.,None,3.639309
2,Please turn on the light.,This suitcase is very light.,None,2.554499
3,She struck a match.,The football match was exciting.,None,2.655155
4,The store will charge your credit card.,The soldier began to charge forward.,None,2.125100
5,Flowers bloom in spring.,The mattress has a broken spring.,None,2.056130


#### • Evaluation with `google-bert/bert-base-uncased` (contextual)

In [28]:
polysemy_dataset.map(vectorize_func_bert).to_pandas().drop(['vec1', 'vec2'], axis=1)

Map: 100%|█████████████████████████████████| 6/6 [00:00<00:00, 29.33 examples/s]


,sentence1,sentence2,score,similarity
0,I deposited money at the bank.,We sat on the bank of the river.,None,3.238695
1,A bat flew out of the cave.,He swung the bat and hit the ball.,None,3.028426
2,Please turn on the light.,This suitcase is very light.,None,2.693022
3,She struck a match.,The football match was exciting.,None,3.246113
4,The store will charge your credit card.,The soldier began to charge forward.,None,2.331136
5,Flowers bloom in spring.,The mattress has a broken spring.,None,2.757084


#### • Evaluation with `sentence-transformers/all-MiniLM-L6-v2` (contextual)

In [29]:
polysemy_dataset.map(vectorize_func_st).to_pandas().drop(['vec1', 'vec2'], axis=1)

Map: 100%|████████████████████████████████| 6/6 [00:00<00:00, 115.69 examples/s]


,sentence1,sentence2,score,similarity
0,I deposited money at the bank.,We sat on the bank of the river.,None,2.073755
1,A bat flew out of the cave.,He swung the bat and hit the ball.,None,2.747482
2,Please turn on the light.,This suitcase is very light.,None,1.096775
3,She struck a match.,The football match was exciting.,None,1.674340
4,The store will charge your credit card.,The soldier began to charge forward.,None,1.080708
5,Flowers bloom in spring.,The mattress has a broken spring.,None,1.065142


Polysemy remains challenging for all embedding models, as the same word can appear in entirely different semantic contexts.

**word2vec-google-news-300** assigns relatively high similarity scores across all examples, reflecting its inability to distinguish between multiple senses of the same word. Since each word has only one fixed embedding, lexical ambiguity directly inflates similarity.

**google-bert/bert-base-uncased** often produces even higher scores than Word2Vec. Although **BERT** is contextual, it is not explicitly optimized for sentence-level similarity, and simple pooling can blur sense distinctions.

**sentence-transformers/all-MiniLM-L6-v2** consistently yields the lowest similarity scores, particularly for strongly ambiguous pairs such as light, charge, and spring. This indicates substantially better contextual disambiguation.

Overall, **Sentence-BERT** demonstrates the strongest robustness to lexical ambiguity, while both **Word2Vec** and vanilla **BERT** tend to overestimate similarity when the same polysemous word appears in both sentences.

## 4. Complex Syntactic Structures

Complex syntactic structures include long sentences, nested clauses, passive constructions, and intricate dependency relationships. Correctly interpreting such sentences requires sensitivity to grammatical structure and compositional meaning.

In [30]:
# these examples were taken from df
complex_synt_strct = [
    {'id': 3703, 
     'sentence1':
         "iraq has been lobbying for the security council to stop using the country's \
          oil revenue to pay compensation to victims of the 1991 gulf war and the salaries of the united nations monitoring, \
          verification and inspection commission inspectors and to have all money remaining in the united nation's oil-for-food accounts \
          transferred to the government's development fund. ", 
     'sentence2': 
         "iraq's new leaders have been lobbying for the united nations security council to stop using the iraq's oil \
          revenue to pay the salaries of the inspectors and to have all money remaining in the united nation's oil-for-food account \
          transferred to the iraqi government.", 
    'score': 4},
    
    {'id': 3492, 
     'sentence1': 
         "russia ratified the updated treaty in 2004 but the united states and other nato members have \
          refused to do so arguing that moscow must first fulfill obligations to withdraw forces from georgia \
          and from moldova's separatist region of trans-dniester. ", 
     'sentence2': 
         "russia has ratified the amended version but the united states and other nato members have refused \
          to do so until the russian government withdraws troops from the former soviet republics of moldova and georgia.",
     'score': 4.2},
    {'id': 3737, 
     'sentence1': 
         'the resolution requires all 192 united nations member states to adopt laws to prevent terrorists, black marketeers \
          and other non-state actors from manufacturing, acquiring or trafficking in nuclear, biological or chemical weapons \
          or the materials to make them. ', 
     'sentence2': 
         'resolution 1540 requires all countries to adopt laws to prevent non-state actors from manufacturing, acquiring \
          or trafficking in nuclear, biological or chemical weapons, the materials to make them, and the missiles and other \
          systems to deliver them.', 
     'score': 3.6},
    {'id': 3564, 
     'sentence1': 
         'estonian officials stated that some of the cyber attacks that caused estonian government websites to shut \
          down temporarily came from computers in the administration of russia including in the office of president vladimir putin. ', 
         'sentence2': 'officials in estonia including prime minister andrus ansip have claimed that some of the cyber attacks came \
         from russian government computers including computers in the office of russian president vladimir putin.', 
     'score': 3.8}
]

complex_synt_strct_dataset = Dataset.from_list(complex_synt_strct)
pd.DataFrame(complex_synt_strct)

,id,sentence1,sentence2,score
0,3703,iraq has been lobbying for the security counci...,iraq's new leaders have been lobbying for the ...,4.0
1,3492,russia ratified the updated treaty in 2004 but...,russia has ratified the amended version but th...,4.2
2,3737,the resolution requires all 192 united nations...,resolution 1540 requires all countries to adop...,3.6
3,3564,estonian officials stated that some of the cyb...,officials in estonia including prime minister ...,3.8


#### • Evaluation with `word2vec-google-news-300` (static)

In [31]:
complex_synt_strct_dataset.map(vectorize_and_get_similarity_static).to_pandas().drop(['vec1', 'vec2'], axis=1)

Map: 100%|████████████████████████████████| 4/4 [00:00<00:00, 745.12 examples/s]


,id,sentence1,sentence2,score,similarity
0,3703,iraq has been lobbying for the security counci...,iraq's new leaders have been lobbying for the ...,4.0,4.731268
1,3492,russia ratified the updated treaty in 2004 but...,russia has ratified the amended version but th...,4.2,4.594383
2,3737,the resolution requires all 192 united nations...,resolution 1540 requires all countries to adop...,3.6,4.766376
3,3564,estonian officials stated that some of the cyb...,officials in estonia including prime minister ...,3.8,4.467106


#### • Evaluation with `google-bert/bert-base-uncased` (contextual)

In [32]:
complex_synt_strct_dataset.map(vectorize_func_bert).to_pandas().drop(['vec1', 'vec2'], axis=1)

Map: 100%|█████████████████████████████████| 4/4 [00:00<00:00, 11.04 examples/s]


,id,sentence1,sentence2,score,similarity
0,3703,iraq has been lobbying for the security counci...,iraq's new leaders have been lobbying for the ...,4.0,4.842567
1,3492,russia ratified the updated treaty in 2004 but...,russia has ratified the amended version but th...,4.2,4.751762
2,3737,the resolution requires all 192 united nations...,resolution 1540 requires all countries to adop...,3.6,4.843889
3,3564,estonian officials stated that some of the cyb...,officials in estonia including prime minister ...,3.8,4.727144


#### • Evaluation with `sentence-transformers/all-MiniLM-L6-v2` (contextual)

In [33]:
complex_synt_strct_dataset.map(vectorize_func_st).to_pandas().drop(['vec1', 'vec2'], axis=1)

Map: 100%|█████████████████████████████████| 4/4 [00:00<00:00, 49.19 examples/s]


,id,sentence1,sentence2,score,similarity
0,3703,iraq has been lobbying for the security counci...,iraq's new leaders have been lobbying for the ...,4.0,4.357082
1,3492,russia ratified the updated treaty in 2004 but...,russia has ratified the amended version but th...,4.2,4.161592
2,3737,the resolution requires all 192 united nations...,resolution 1540 requires all countries to adop...,3.6,3.539864
3,3564,estonian officials stated that some of the cyb...,officials in estonia including prime minister ...,3.8,4.076159


In this experiment, **Word2Vec** produced generally high similarity scores, but its behaviour is largely driven by shared keywords and does not reflect the underlying syntactic complexity of the sentences. It treats structurally different but lexically similar sentences as almost equivalent.

**BERT** showed slightly improved consistency, benefiting from contextual representations that better capture relationships between words in longer sentences. However, it still does not explicitly model sentence-level similarity, leading to limited sensitivity to structural variation.

**Sentence-BERT (all-MiniLM-L6-v2)** demonstrated the most balanced performance, producing scores that are more stable and better aligned with semantic similarity despite syntactic changes. It appears more robust to variations in sentence structure while preserving overall meaning.

## 5. Out-of-Vocabulary (OOV) Terms

Out-of-vocabulary terms are words that were not observed during pretraining. They commonly include rare names, newly coined terms, domain-specific jargon, and typographical variations.

Robust handling of OOV terms is crucial for real-world NLP applications.

> **Remark:** In static embedding models such as **word2vec-google-news-300**, each word is mapped to a single fixed vector. If a word is not present in the vocabulary, it cannot be represented at all, which makes OOV a direct and important source of performance degradation.
    In contrast, contextual models such as **google-bert/bert-base-uncased** and **sentence-transformers/all-MiniLM-L6-v2** do not rely on a fixed word vocabulary. Instead, they use subword tokenization (e.g., **WordPiece**), which decomposes unseen words into smaller known units. As a result, classical OOV does not occur in these models.
    Therefore, OOV analysis is meaningful only for static embeddings

#### • Evaluation with `word2vec-google-news-300` (static)

In [34]:
oov_df = df.copy()
oov_df.join(static_vectors.to_pandas()['similarity'], how='inner')

,split,genre,dataset,year,sid,score,sentence1,sentence2,similarity
0,train,main-captions,MSRvid,2012test,1,5.00,A plane is taking off.,An air plane is taking off.,4.434653
1,train,main-captions,MSRvid,2012test,4,3.80,A man is playing a large flute.,A man is playing a flute.,4.727526
2,train,main-captions,MSRvid,2012test,5,3.80,A man is spreading shreded cheese on a pizza.,A man is spreading shredded cheese on an uncoo...,4.459021
3,train,main-captions,MSRvid,2012test,6,2.60,Three men are playing chess.,Two men are playing chess.,4.975553
4,train,main-captions,MSRvid,2012test,9,4.25,A man is playing the cello.,A man seated is playing the cello.,4.623239
...,...,...,...,...,...,...,...,...,...
5744,train,main-news,headlines,2016,1456,0.00,Severe Gales As Storm Clodagh Hits Britain,Merkel pledges NATO solidarity with Latvia,0.907577
5745,train,main-news,headlines,2016,1465,0.00,Dozens of Egyptians hostages taken by Libyan t...,Egyptian boat crash death toll rises as more b...,2.265827
5746,train,main-news,headlines,2016,1466,0.00,President heading to Bahrain,President Xi: China to continue help to fight ...,1.931352
5747,train,main-news,headlines,2016,1470,0.00,"China, India vow to further bilateral ties",China Scrambles to Reassure Jittery Stock Traders,1.937988


In [35]:
from gensim.downloader import load
import re

# Build a vocabulary set from the pretrained Word2Vec model
# This allows efficient O(1) lookup for each token
vocab = set(word2vec_model.key_to_index)


# Compute the proportion of out-of-vocabulary (OOV) words.
def oov_ratio(sentence):
    # Simple regex-based tokenization
    words = re.findall(r"\b\w+\b", sentence.lower())
    # Avoid division by zero for empty sentences
    if not words:
        return 0.0
    # Count words absent from the pretrained vocabulary
    missing = sum(word not in vocab for word in words)
    # Return OOV ratio
    return missing / len(words)


oov_df = df.copy()
# Add Word2Vec similarity predictions
oov_df = oov_df.join(
    static_vectors.to_pandas()["similarity"],
    how="inner"
)
# Compute OOV ratio for each sentence separately
oov_df["oov1"] = oov_df["sentence1"].apply(oov_ratio)
oov_df["oov2"] = oov_df["sentence2"].apply(oov_ratio)

# Maximum OOV ratio across the sentence pair
# Useful for identifying the harder sentence
oov_df["oov_max"] = oov_df[["oov1", "oov2"]].max(axis=1)

# Average OOV ratio across both sentences
# Serves as an overall lexical rarity score
oov_df["oov_avg"] = (oov_df["oov1"] + oov_df["oov2"]) / 2

# Display the most relevant columns
oov_df[
    [
        "sentence1",
        "sentence2",
        "score",
        "similarity",
        "oov1",
        "oov2",
        "oov_avg",
    ]
].head(10)

,sentence1,sentence2,score,similarity,oov1,oov2,oov_avg
0,A plane is taking off.,An air plane is taking off.,5.00,4.434653,0.200000,0.000000,0.100000
1,A man is playing a large flute.,A man is playing a flute.,3.80,4.727526,0.285714,0.333333,0.309524
2,A man is spreading shreded cheese on a pizza.,A man is spreading shredded cheese on an uncoo...,3.80,4.459021,0.333333,0.100000,0.216667
3,Three men are playing chess.,Two men are playing chess.,2.60,4.975553,0.000000,0.000000,0.000000
4,A man is playing the cello.,A man seated is playing the cello.,4.25,4.623239,0.166667,0.142857,0.154762
5,Some men are fighting.,Two men are fighting.,4.25,4.693394,0.000000,0.000000,0.000000
6,A man is smoking.,A man is skating.,0.50,2.694183,0.250000,0.250000,0.250000
7,The man is playing the piano.,The man is playing the guitar.,1.60,4.682547,0.000000,0.000000,0.000000
8,A man is playing on a guitar and singing.,A woman is playing an acoustic guitar and sing...,2.20,4.517063,0.333333,0.222222,0.277778
9,A person is throwing a cat on to the ceiling.,A person throws a cat on the ceiling.,5.00,4.652732,0.300000,0.250000,0.275000


In [36]:
oov_df["error"] = abs(oov_df["similarity"] - oov_df["score"]) # Measures how far the model's predicted similarity
# Group sentence pairs into five OOV ranges and aggregate by mean and count
oov_df.groupby(pd.cut(oov_df["oov_avg"], bins=5)).agg({
    'error': ['mean', 'count']})

/var/folders/vf/b67_gn4570g19xv1l1ckmqnm0000gp/T/ipykernel_10560/364000219.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  oov_df.groupby(pd.cut(oov_df["oov_avg"], bins=5)).agg({


error      
                        mean count
oov_avg                           
(-0.000675, 0.135]  1.260271  2812
(0.135, 0.27]       1.328187  2183
(0.27, 0.405]       1.407523   723
(0.405, 0.54]       1.187353    30
(0.54, 0.675]       0.550408     1

In [37]:
pearsonr(oov_df["oov_max"], oov_df["error"])

PearsonRResult(statistic=0.055893909433978525, pvalue=2.2313140385620525e-05)

OOV analysis for **word2vec-google-news-300** reveals a weak but statistically significant relationship between vocabulary coverage and prediction error.

For the three largest OOV groups, the average error steadily increases from 1.26 to 1.41 as the OOV ratio rises. This trend is supported by a substantial number of examples in each bin, making it reliable. The final two bins contain only 30 and 1 samples, respectively, so their lower error should not be overinterpreted.

This pattern indicates that **Word2Vec** becomes less accurate as the proportion of unseen words grows. However, the effect remains modest because the STS Benchmark contains relatively few true OOV-heavy examples.

Overall, the results confirm that static embeddings are sensitive to vocabulary coverage, while also highlighting that extreme OOV cases are underrepresented in standard semantic similarity benchmarks.